# Model Evaluation: Predicting Conflict Escalation

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B:** Baseline + Text embeddings

There are four different versions of Model B evaluated throughtout this notebook.
* All events, no PCA (~790 features) `all_nopca`
* All events, PCA (~73 features) `all_pca`
* Conflict-only events, no PCA (~790 features) `conflict_nopca`
* Conflict-only events, PCA (~47 features) `conflict_pca`

This notebook compares the results from the best models where `k`=1.75 and the threshold fix has been applied, as dicussed in the methodology decisions notebook.

Each model configuration has a number of recorded results on:
* Train (2018-2022) - period of time where there is no civil war. Results on training-CV splits. 
* Onset (2023) - including the three months where civil war esclated (April 2023)
* Active (2024-2025) - full period of time with ongoing active civil war

Results are read in from:
`sudan_resuluts.csv` - evaluation results file.
`sudan_results_seeds.xlxs` - file used to test different hyperparameters to stress test findings.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from models.run_best_models import run_best_models
from utils.reporting import read_model_reports

In [2]:
# Reading and filter results
def apply_config(df: pd.DataFrame, config) -> pd.DataFrame:
    """Filter the results for the decided config."""
    mask = pd.Series(True, index=df.index)
    for col, val in config.items():
        mask &= df[col] == val

    food_ok = (df["include_food"] == False) | (df["price_recency"] == True)
    mask &= food_ok
    return df[mask].copy()


def variant_label(row):
    if row["include_text"] == False:
        return "model_a"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == True:
        return "conflict_pca"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == False:
        return "conflict_nopca"
    if row["conflict_only_embeddings"] == False and row["use_pca"] == True:
        return "all_pca"
    return "all_nopca"


def filter_results(all_results, config):
    results = apply_config(all_results, config)
    results["variant"] = results.apply(variant_label, axis=1)
    return results

In [3]:
# Baseline summary tables

METRICS = {
    "train": ["train_cv_aupr", "train_cv_f1"],
    "onset": ["onset_aupr", "onset_f1_class1"],
    "active": ["active_aupr", "active_f1_class1"],
}


def performance_summary(results, variants, split=None):
    """Calculates aggregate performance statistics for the given model variant(s)."""
    metrics = (
        METRICS[split]
        if split is not None
        else [m for split_metrics in METRICS.values() for m in split_metrics]
    )

    subset = results[results["variant"].isin(variants)]

    rows = []
    for variant, grp in subset.groupby("variant"):
        if len(grp) == 1:
            summary = grp[metrics].round(4).T
            summary.columns = ["value"]
        else:
            summary = (
                grp[metrics].agg(["mean", "median", "min", "max", "std"]).round(4).T
            )
        summary.insert(0, "n_configs", len(grp))
        summary.index = summary.index.str.replace("_class1", "")
        summary.insert(0, "variant", variant)
        rows.append(summary)

    return pd.concat(rows).set_index("variant", append=True).swaplevel().sort_index()


def baseline_comparison_table(results, split=None):
    """Match text-augmented configs to their baseline (Model A) counterpart and compare metrics.

    If `split` is given (one of "train", "onset", "active"), only that split's metrics
    (METRICS[split]) are compared. If omitted, metrics for every split in METRICS are compared.
    """
    if split is not None:
        metrics = METRICS[split]
        recall_col = f"{split}_recall_class1"
    else:
        metrics = [m for split_metrics in METRICS.values() for m in split_metrics]
        recall_col = None

    cols_to_convert = metrics + (
        [recall_col] if recall_col and recall_col in results.columns else []
    )
    for m in cols_to_convert:
        results[m] = pd.to_numeric(results[m], errors="coerce")

    key_cols = [
        "include_food",
        "include_rain",
        "k",
        "event_col",
        "n_splits",
    ]

    baseline = results[results["include_text"] == False].set_index(key_cols)
    text_df = results[results["include_text"] == True].copy()

    results_comparison = []
    for variant, grp in text_df.groupby("variant"):
        grp = grp.set_index(key_cols)

        join_cols = list(
            dict.fromkeys(
                metrics
                + ([recall_col] if recall_col and recall_col in grp.columns else [])
            )
        )
        matched = grp.join(baseline[join_cols], rsuffix="_base", how="inner")

        mean_preds = (
            matched["n_predictors"].mean()
            if "n_predictors" in matched.columns
            else float("nan")
        )

        collapse_rate = (
            (matched[recall_col] > 0.9).mean() * 100
            if recall_col and recall_col in matched.columns
            else float("nan")
        )

        row = {
            "variant": variant,
            "n_matched_pairs": len(matched),
            "mean_n_predictors": round(mean_preds, 1),
            "percent_collapsed_recall": round(collapse_rate, 1),
        }

        for m in metrics:
            diff = matched[m] - matched[f"{m}_base"]
            row[f"mean_{m}"] = round(matched[m].mean(), 4)
            row[f"mean_{m}_baseline"] = round(matched[f"{m}_base"].mean(), 4)
            row[f"percent_better_baseline_{m}"] = round((diff > 0).mean() * 100, 1)
            row[f"mean_diff_{m}"] = round(diff.mean(), 4)
            row[f"median_diff_{m}"] = round(diff.median(), 4)

        results_comparison.append(row)

    results_table = pd.DataFrame(results_comparison).set_index("variant")
    display(results_table)


# Recall/precision table
def recall_precision_summary(results, split):
    recall_col = f"{split}_recall_class1"
    precision_col = f"{split}_precision_class1"

    results_rp = results.copy()
    results_rp["collapsed"] = results_rp[recall_col] > 0.9

    return (
        results_rp.groupby("variant")
        .agg(
            n_configs=(recall_col, "count"),
            mean_recall=(recall_col, "mean"),
            mean_precision=(precision_col, "mean"),
            collapse_rate=("collapsed", "mean"),
        )
        .round(3)
    )

In [4]:
# Table for comparing event columns
def event_col_preference_table(results, metrics):
    match_cols = ["include_food", "include_rain", "n_splits", "variant"]

    sub = results[results["event_col"] == "sub_event_type"].set_index(match_cols)
    evt = results[results["event_col"] == "event_type"].set_index(match_cols)

    rows = []
    for metric in metrics:
        paired = sub[[metric]].join(
            evt[[metric]], lsuffix="_sub", rsuffix="_evt", how="inner"
        )
        for variant, grp in paired.reset_index().groupby("variant"):
            n_sub_wins = (grp[f"{metric}_sub"] > grp[f"{metric}_evt"]).sum()
            n_evt_wins = (grp[f"{metric}_evt"] > grp[f"{metric}_sub"]).sum()
            n_total = len(grp)
            rows.append(
                {
                    "variant": variant,
                    "metric": metric,
                    "n_pairs": n_total,
                    "sub_event_type_wins": n_sub_wins,
                    "event_type_wins": n_evt_wins,
                }
            )

    table = pd.DataFrame(rows)

    totals = table.groupby("metric")[
        ["n_pairs", "sub_event_type_wins", "event_type_wins"]
    ].sum()
    totals["variant"] = "TOTAL"
    totals = totals.reset_index()
    print(totals)

    return pd.concat([table, totals], ignore_index=True)

# Introduction
Predicting the exact outbreak of a rare, unprecedented civil war is an inherently difficult forecasting task. Before evaluating the impact of text embeddings, it is important to establish the predictive baseline of Model A, which relies purely on structural data: historical tabular ACLED counts, food prices, and rainfall.

Across 16 tested configurations Model A has a **mean train-CV AUPR of 0.2388** and **F1 of 0.3408**.

While these absolute metrics are lower than standard machine learning benchmarks, this reflects both the limited scope of this project and the challenge of training data in conflict prediction. As dicussed in the main project report, comparable conflict forecasting models take a multi-country approach which both expands the available training data (including available structural variables) and the number of conflict escalations for the model to learn from. The training data, while it includes conflict escalations it does not include the type of esclation it is trying to predict. The objective of this evaluation project is not to present a flawless predictive system, but to determine whether text embeddings can overcome the limitations of structural data to provide an earlier, measurable warning signal.

In [5]:
part1_config = {"k": 1.75, "threshold_fix_applied": True, "seed": 23}

ALL_RESULTS = pd.read_csv("evaluation/sudan_results.csv")
PART1_RESULTS = filter_results(ALL_RESULTS, part1_config)

In [6]:
performance_summary(PART1_RESULTS, ["model_a"])

n_configs    mean  median     min     max     std
variant                                                                 
model_a active_aupr           16  0.2267  0.2218  0.1899  0.2785  0.0213
        active_f1             16  0.2586  0.2564  0.2235  0.2985  0.0240
        onset_aupr            16  0.3198  0.3138  0.2700  0.3738  0.0316
        onset_f1              16  0.3224  0.3212  0.2192  0.4000  0.0424
        train_cv_aupr         16  0.2388  0.2356  0.2154  0.2724  0.0163
        train_cv_f1           16  0.3408  0.3366  0.3184  0.3683  0.0180

# Part 1 - does text improve model performance?
Part 1 looks at the question of 'does text improve performance?' rather than comparing the final chosen best models. The following things have been fixed (see methdodology decisions for discussion):

* `k`=1.75 - this is fixed as it defines the prediction target.
* The threshold fix has been applied - this was a bug fix that has been resolved. 
* Food price recency flag has been included - original runs (not included in these results) did not include the price-recency flag for food price data and instead relied on forward fill. More on how zero values were filled is dicussed in the methodology notebook.

The following implementation choices are allowed to roam in part 1 as they are do not change the underlying task, only what inputs the model draws on and how it is fit:
* `n_splits`
* `event_col`
* The inclusion of additional strucutral variables (food/rain)

# 1.1 Training performance
The results on the cross-validation training splits give an indication on whether the models are overfitting to the training data and would therefore not be generalisable.

As the aim is to understand if adding text improves the baseline (structural features only), each baseline and text-added configuration was compared on training-CV AUPR and F1. In the initial evaluation stage, the percetange of times the text model (matched to the same baseline configuration) beats baseline is evaluated. 

Evaluating model performance across the peaceful baseline period reveals that *text embeddings degrade model performance relative to structural features*. On AUPR, only two of the four variants ever beat baseline, and only barely (four models total).

* Neither all-event (`all_nopca` and `all_pca`) variant beats baseline on AUPR in any comparison. 
* Conflict-only text, both with (`conflict_pca`) and without PCA (`conflict_nopca`), is tied as the best performing model on training AUPR, each beating baseline in 2 of 16 comparisons (12.5%). 
* On F1, conflict-only text without PCA (`conflict_nopca`) is the clearer of the two, beating baseline in 4 of 16 comparisons (25%) against conflict-only PCA's 2 of 16 (12.5%), so it's the stronger overall performer of the two once both metrics are considered, but the AUPR result on its own does not distinguish between them.
  
In a standard machine learning pipeline, discarding a model that loses across 100% of training folds would be best practice.

**Baseline vs best performing model on train-cv**

| Model Variant | Mean Train CV AUPR | Mean Train CV F1 | AUPR Win Rate vs. Baseline | F1 Win Rate vs. Baseline |
| :--- | :---: | :---: | :---: | :---: |
| **Model A (`model_a`)** | **0.2388** | **0.3408** | — | — |
| **Model B (`conflict_nopca`)** | 0.2267 | 0.3235 | 12.5% | 25.0% |




In [7]:
# Baseline comparison for training AUPR and F1
baseline_comparison_table(PART1_RESULTS, "train")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_train_cv_aupr,mean_train_cv_aupr_baseline,percent_better_baseline_train_cv_aupr,mean_diff_train_cv_aupr,median_diff_train_cv_aupr,mean_train_cv_f1,mean_train_cv_f1_baseline,percent_better_baseline_train_cv_f1,mean_diff_train_cv_f1,median_diff_train_cv_f1
variant,,,,,,,,,,,,,
all_nopca,16,790.5,NaN,0.1993,0.2388,0.0,-0.0395,-0.0374,0.2881,0.3408,0.0,-0.0526,-0.0473
all_pca,16,72.5,NaN,0.2049,0.2388,0.0,-0.0339,-0.0368,0.2943,0.3408,0.0,-0.0465,-0.0431
conflict_nopca,16,790.5,NaN,0.2267,0.2388,12.5,-0.0121,-0.0130,0.3235,0.3408,25.0,-0.0173,-0.0184
conflict_pca,16,46.5,NaN,0.2255,0.2388,12.5,-0.0133,-0.0160,0.3225,0.3408,12.5,-0.0183,-0.0182


# 1.2 Onset performance


During the 2023 onset period, text features do provide a performance boost over baseline, but identifying the "best" text variant depends on whether evaluation prioritises AUPR or F1. This metric split aligns strictly with corpus selection (conflict-only vs. all-events) rather than dimensionality reduction (PCA).

*Conflict-only text*

Both conflict-only variants outperform both all-event variants on AUPR:
* Conflict-only without PCA (`conflict_nopca`): 75.0% of 16 matched pairs modestly beat baseline. With a mean AUPR diff of +0.017 above baseline.
* Conflict-only with PCA (`conflict_pca`): 68.8% of 16 models beat baseline. With +0.009 mean AUPR diff above baseline. 
In this model's context, AUPR demonstrates how well the model balances the accuracy of the region-months it flags as escalations (precision) and its ability to find the actual escalations (recall) across all confidence thresholds. 


*All-event text*

In contrast both all-event variants outperform both conflict-only variants on F1:
* All-event non-PCA (`all_nopca`): Train-CV F1 of 87.5% of baseline models, mean diff +0.032 above baseline
* All-event PCA (`all_pca`): Train-CV F1 of 43.8%, mean diff -0.005 below baseline. 
The shows the all-event model assigns higher esclation probabilities to pre-conflict regions which causes recall to increase, but it lacks precision. The all-event model assigns higher escalation probabilities to pre-conflict regions. At the calibrated F1 threshold, this causes recall to surge—jumping from ~25% in conflict-only models up to 75%–88% in the all-event model. Because conflict-only models are too conservative and miss most onset events (low recall cripples their F1), the massive recall boost in all_nopca easily outweighs its moderate penalty in precision, pushing its overall F1 score significantly higher.

All-event text with PCA is the weakest text variant on AUPR specifically. Though it catches more escalations (better recall), it's precicision is the lowest of the models. It is  the only one with a negative mean difference on AUPR (-0.0085), and the only one that fails to beat baseline in a majority of comparisons on either metric.

| Model Variant | AUPR Win Rate | Mean AUPR Diff | F1 Win Rate | Mean F1 Diff | Mean Recall | Mean Precision | Collapse Rate |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model A (Baseline)** | — | — | — | — | 38.4% | 29.3% | **0.0%** |
| **`conflict_nopca`** | **75.0%** | **+0.0174** | 25.0% | -0.0217 | 25.0% | **45.4%** | **0.0%** |
| **`conflict_pca`** | 68.8% | +0.0090 | 12.5% | -0.0308 | 25.6% | 40.5% | **0.0%** |
| **`all_nopca`** | 56.2% | +0.0038 | **87.5%** | **+0.0318** | **75.0%** | 23.5% | 12.5% |
| **`all_pca`** | 43.8% | -0.0085 | 43.8% | -0.0050 | 42.5% | 29.2% | 6.2% |

The difference between this variation isn't two different directions by chance. Conflict-only text runs a much more conservative operating point, it ranks escalation months well overall (hence the stronger AUPR across both its PCA and non-PCA forms), but is cautious about calling any specific month an escalation. All-event text runs the opposite, flagging more broadly, which drives its F1 advantage but also means two of all-event non-PCA's 16 configurations collapse to predicting esclations for almost everything (`onset_recall_class1 > 0.9`). Given that a missed escalation is treated as the more costly error for an early-warning system in this project's framing, all-event text's recall-leaning approach is not obviously the wrong choice, but it should be considered a trade-off.


In [8]:
# Onset comparison to baseline
baseline_comparison_table(PART1_RESULTS, "onset")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_onset_aupr,mean_onset_aupr_baseline,percent_better_baseline_onset_aupr,mean_diff_onset_aupr,median_diff_onset_aupr,mean_onset_f1_class1,mean_onset_f1_class1_baseline,percent_better_baseline_onset_f1_class1,mean_diff_onset_f1_class1,median_diff_onset_f1_class1
variant,,,,,,,,,,,,,
all_nopca,16,790.5,12.5,0.3236,0.3198,56.2,0.0038,0.0063,0.3542,0.3224,87.5,0.0318,0.0310
all_pca,16,72.5,6.2,0.3113,0.3198,43.8,-0.0085,-0.0054,0.3174,0.3224,43.8,-0.0050,-0.0022
conflict_nopca,16,790.5,0.0,0.3372,0.3198,75.0,0.0174,0.0202,0.3007,0.3224,25.0,-0.0217,-0.0362
conflict_pca,16,46.5,0.0,0.3289,0.3198,68.8,0.0090,0.0038,0.2916,0.3224,12.5,-0.0308,-0.0385


In [9]:
# Onset precision and recall
recall_precision_summary(PART1_RESULTS, "onset")

,n_configs,mean_recall,mean_precision,collapse_rate
variant,,,,
all_nopca,16,0.750,0.235,0.125
all_pca,16,0.425,0.292,0.062
conflict_nopca,16,0.250,0.454,0.000
conflict_pca,16,0.256,0.405,0.000
model_a,16,0.384,0.293,0.000


**Removing collapsed models**

To test whether the performance gains of `all_nopca` were artificially driven by the 12.5% threshold collapse rate, the matched baseline comparison was re-run on  non-collapsed configurations (`onset_recall < 0.9`).

Both all-event text variants sit above baseline on recall and below it on precision. Both conflict-only text variants sit below baseline on recall and above it on precision. 

`all_nopca` is the extreme case at both ends. It has nearly double baseline's recall, and the only variant with a high collapse rate (12.5%) when setting the collapse threshold to `recall > 0.9`. `all_pca` shows recall in the same direction but far more mildly (recall 0.425 vs baseline's 0.384, a modest lean rather than a strong one), and its 6.2% collapse rate (1 of 16 configs) it noteworthy but not the dominant outcome. 

In comparison to the baseline:

* **F1** `all_nopca` retains its F1 advantage, beating the structural baseline in **85.7% of matched pairs** (12/14 runs) with a mean F1 gain of **+0.0299** (median **+0.0310**).
* **Recall** Non-collapsed `all_nopca` recall settles at **72.6%**, remaining nearly 2x higher than the baseline (38.2%) and 3x higher than conflict-only text (25.0%).
* **AUPR Trade-off:** As expected, `all_nopca` mean AUPR difference becomes neutral (**-0.0007**, median **+0.0010**), reinforcing that `conflict_nopca` remains the optimal variant for threshold-agnostic probability ranking, while `all_nopca` provides the superior discrete early-warning alarm.

In [10]:
# Recall and precision for non-collapsed models
results_no_collapse = PART1_RESULTS[PART1_RESULTS["onset_recall_class1"] < 0.9].copy()
recall_precision_summary(results_no_collapse, "onset")

,n_configs,mean_recall,mean_precision,collapse_rate
variant,,,,
all_nopca,14,0.726,0.237,0.0
all_pca,15,0.390,0.296,0.0
conflict_nopca,16,0.250,0.454,0.0
conflict_pca,16,0.256,0.405,0.0
model_a,16,0.384,0.293,0.0


In [11]:
# Comparison to baseline for non-collapsed models
baseline_comparison_table(results_no_collapse, "onset")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_onset_aupr,mean_onset_aupr_baseline,percent_better_baseline_onset_aupr,mean_diff_onset_aupr,median_diff_onset_aupr,mean_onset_f1_class1,mean_onset_f1_class1_baseline,percent_better_baseline_onset_f1_class1,mean_diff_onset_f1_class1,median_diff_onset_f1_class1
variant,,,,,,,,,,,,,
all_nopca,14,791.8,0.0,0.3232,0.3239,50.0,-0.0007,0.0010,0.3537,0.3238,85.7,0.0299,0.0310
all_pca,15,73.3,0.0,0.3112,0.3211,40.0,-0.0100,-0.0082,0.3129,0.3218,40.0,-0.0089,-0.0033
conflict_nopca,16,790.5,0.0,0.3372,0.3198,75.0,0.0174,0.0202,0.3007,0.3224,25.0,-0.0217,-0.0362
conflict_pca,16,46.5,0.0,0.3289,0.3198,68.8,0.0090,0.0038,0.2916,0.3224,12.5,-0.0308,-0.0385


# 1.3 Active performance

*Once the war is underway, the baseline model prevails almost everywhere*. All four text variants now show negative mean AUPR differences against baseline.

Three of the four text variants perform very poorly during active conflict, and all show negative mean differences on both AUPR and F1. Once conflict has been running for months, region-month event counts stop being zero-inflated and the autoregressive/structural features are doing the actual work. The text embeddings aren't acting as a precursor signal the way they might pre-escalation, they're mostly extra dimensions that can cause overfitting.

During active conflict, the text model all-text with PCA is the best performing against baseline. During active conflict dimensionality reduction appears to specifically help text remain useful once conflict is underway. However, the same compression only works slightly on the conflict-only text model during active war. The pattern suggests it isn't PCA alone or text alone driving the active-period result, but the combination of the full event corpus (not just conflict events) compressed down to a smaller, less overfitting-prone feature set.

| Model Variant | Mean Active AUPR | AUPR Win Rate vs. Base | Mean Active F1 | F1 Win Rate vs. Base | Mean Recall | Mean Precision |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model A (Baseline)** | **0.2267** | — | **0.2586** | — | 41.4% | **20.8%** |
| **`all_pca`** | 0.2180 | 25.0% | 0.2453 | **50.0%** | 43.4% | 20.4% |
| **`conflict_pca`** | 0.1894 | 6.2% | 0.1397 | 6.2% | 15.8% | 20.6% |
| **`conflict_nopca`** | 0.1672 | 0.0% | 0.1028 | 0.0% | 10.7% | 18.3% |
| **`all_nopca`** | 0.1558 | **0.0%** | 0.2399 | 25.0% | **83.5%** | 14.1% |

In [12]:
baseline_comparison_table(PART1_RESULTS, "active")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_active_aupr,mean_active_aupr_baseline,percent_better_baseline_active_aupr,mean_diff_active_aupr,median_diff_active_aupr,mean_active_f1_class1,mean_active_f1_class1_baseline,percent_better_baseline_active_f1_class1,mean_diff_active_f1_class1,median_diff_active_f1_class1
variant,,,,,,,,,,,,,
all_nopca,16,790.5,56.2,0.1558,0.2267,0.0,-0.0708,-0.0703,0.2399,0.2586,25.0,-0.0187,-0.0097
all_pca,16,72.5,12.5,0.2180,0.2267,25.0,-0.0087,-0.0107,0.2453,0.2586,50.0,-0.0133,0.0032
conflict_nopca,16,790.5,0.0,0.1672,0.2267,0.0,-0.0595,-0.0594,0.1028,0.2586,0.0,-0.1557,-0.1666
conflict_pca,16,46.5,0.0,0.1894,0.2267,6.2,-0.0373,-0.0359,0.1397,0.2586,6.2,-0.1188,-0.0960


In [13]:
# TODO inset a chart here

# Part 2 - choosing the best model
For part two, the single best model configuration for the baseline model (Model A) will be comapred against the best model configs for the text variants. In order to make a fair comparison, configurations that were allowed to run freely in part 1 have been limited.

# 2.1 Defining the final config

## 2.1.1 Setting included data

In part 1 dropping rain or dropping food, sometimes scores marginally higher for some variants. Deliberately not following that signal here is the point, if the final comparison quietly adopted whichever configuration flattered each model, the comparison would no longer be about text, it would be about whichever feature set happened to win, model by model. 

For the final model comparison, **food price features are retained while rainfall features are dropped globally**. This decision is grounded in three key empirical and domain findings:
* **AUPR degradation**: Including rainfall data appears to hurt model performance in all models. In the baseline model, including rain lowers train-cv AUPR from 0..2492 to 0.2285.
* **SHAP feature importance**: SHAP feature importance analysis shows that the models heavily downweight rainfall - it accounts for only 6.3%-8.3% of predictive weight in training. 

Conversely, food price commodities and the `months_since_reading_*` price recency indicator are retained because they capture real-time supply chain disruptions and local economic shocks directly tied to conflict escalation. 

In [14]:
rain_summary = (
    PART1_RESULTS.groupby(["variant", "include_rain"])["train_cv_aupr"].mean().round(4)
)

no_rain = rain_summary.xs(False, level="include_rain")
with_rain = rain_summary.xs(True, level="include_rain")

rain_comparison_table = pd.DataFrame(
    {
        "Train-CV (No Rain) mean AUPR": no_rain,
        "Train-CV (With Rain) mean AUPR": with_rain,
        "TrainCV diff mean AUPR": with_rain - no_rain,
    }
).round(4)

display(rain_comparison_table)

,Train-CV (No Rain) mean AUPR,Train-CV (With Rain) mean AUPR,TrainCV diff mean AUPR
variant,,,
all_nopca,0.2032,0.1955,-0.0077
all_pca,0.2058,0.2040,-0.0018
conflict_nopca,0.2300,0.2234,-0.0066
conflict_pca,0.2325,0.2186,-0.0139
model_a,0.2492,0.2285,-0.0207


## 2.1.2 Setting cross-validation folds (`n_splits = 5`)

The `n_splits` parameter determines the number of expanding-window cross-validation folds used during hyperparameter tuning (testing both `n=4` and `n=5` across models).

Across models, cross-validation training AUPR scores between the two choices are almost identical, showing a small difference of under 0.006 (0.2220 for four splits versus 0.2162 for five splits). Five splits is selected because its structure aligns naturally with the five-year training period (2018–2022), providing intuitive annual expanding increments that maximise training data volume per fold.

In [15]:
PART1_RESULTS.groupby("n_splits")[METRICS["train"]].agg(["mean"]).round(4)

,train_cv_aupr,train_cv_f1
,mean,mean
n_splits,,
4,0.2220,0.3180
5,0.2162,0.3097


In [16]:
PART1_RESULTS["variant"] = PART1_RESULTS.apply(variant_label, axis=1)
PART1_RESULTS.groupby(["n_splits", "variant"])[METRICS["train"]].agg(
    ["mean", "count"]
).round(4)

train_cv_aupr       train_cv_f1      
                                 mean count        mean count
n_splits variant                                             
4        all_nopca             0.2059     8      0.2966     8
         all_pca               0.2124     8      0.3034     8
         conflict_nopca        0.2223     8      0.3189     8
         conflict_pca          0.2289     8      0.3320     8
         model_a               0.2403     8      0.3389     8
5        all_nopca             0.1927     8      0.2797     8
         all_pca               0.1975     8      0.2852     8
         conflict_nopca        0.2311     8      0.3281     8
         conflict_pca          0.2221     8      0.3130     8
         model_a               0.2373     8      0.3427     8

## 2.1.3 Event type (`event_col = `sub_event_type`)
The `event_col` parameter determines whether ACLED event features are chosen from the six events or the 25 sub-event types. Sub-events naturally give the model greater detail but this level of disaggregation reduces the number of positive instances of that sub-event. 

High-level `event-type` categories maintain denser counts, whereas granular `sub_event_type` categories provide the model with richer tactical detail at the cost of increasing sparsity.

The training-cv results demonstrate `sub_event_type`s slight advantage, yielding higher Train CV AUPR in **21 out of 40 paired comparisons** (52.5%). While pretty much a tie in results, domain-knowledge again is important here. The literature indicates that more granular sub-event types can preserve tactical distinctions that broader categories obscure. For example, the high-level `event_type` category "Strategic developments" bundles eight distinct sub-events into a single count, including both property destruction and changes to group activity such as troop movements (ACLED, 2024), collapsing signals that may carry different predictive value for conflict escalation into one figure.

To maximise feature detail and improving performance slightly, `event_col` is set to `sub_event_type` across all baseline and text-augmented configurations. 

In [17]:
event_col_preference_table(PART1_RESULTS, ["train_cv_aupr", "onset_aupr"])

          metric  n_pairs  sub_event_type_wins  event_type_wins variant
0     onset_aupr       40                   30               10   TOTAL
1  train_cv_aupr       40                   21               19   TOTAL


,variant,metric,n_pairs,sub_event_type_wins,event_type_wins
0,all_nopca,train_cv_aupr,8,3,5
1,all_pca,train_cv_aupr,8,6,2
2,conflict_nopca,train_cv_aupr,8,5,3
3,conflict_pca,train_cv_aupr,8,4,4
4,model_a,train_cv_aupr,8,3,5
5,all_nopca,onset_aupr,8,4,4
6,all_pca,onset_aupr,8,4,4
7,conflict_nopca,onset_aupr,8,6,2
8,conflict_pca,onset_aupr,8,8,0
9,model_a,onset_aupr,8,8,0


#### Therefore we set the best model config to the following:

In [18]:
PART2_CONFIG = {
    "k": 1.75,
    "threshold_fix_applied": True,
    "price_recency": True,
    "event_col": "sub_event_type",
    "include_food": True,
    "include_rain": False,
    "n_splits": 5,
}

# 2.2 Final model results (averaged across 5 random-search seeds)

All figures below are the mean across the five random-search seeds tested (23, 32, 111, 999, 2025) at the final configuration set in part 2.1. 

**Training AUPR**: None of the text variants beat Model A's mean training AUPR (0.2286), though `conflict_nopca` comes the closest at 0.2276. 

**Onset AUPR**: both conflict-only text variants beat Model A on average, `conflict_nopca` at 0.3637 and `conflict_pca` at 0.3566 against Model A's 0.339. `conflict_pca` is the more robust of the two: its lowest score across all five seeds (0.341) still exceeds Model A's mean. `conflict_nopca`'s advantage is real on average but noisier, its seed-to-seed standard deviation (0.028) is larger than its mean advantage over Model A (+0.025).

**Onset F1** (threshold-dependent): Model A leads at 0.384, against a best-of-the-rest 0.369 for `all_pca`. Model A's minimum F1 across the five seeds (0.370) still beats every other model's mean.

**Active AUPR**: At a single seed (23), all_pca trails Model A (0.206 vs 0.220) but averaged across all five seeds it edges ahead (0.2157 vs 0.2079). This reversal is fragile, though with the seed-to-seed standard deviation (0.026) over three times the mean advantage (0.008), and all_pca only wins in 2 of the 5 seeds.

In [19]:
PART2_RESULTS = filter_results(ALL_RESULTS, PART2_CONFIG)

In [20]:
metric_cols = [
    "train_cv_aupr",
    "train_cv_f1",
    "onset_aupr",
    "onset_f1_class1",
    "onset_recall_class1",
    "onset_precision_class1",
    "active_aupr",
    "active_f1_class1",
    "active_recall_class1",
    "active_precision_class1",
]

main_metrics_df = PART2_RESULTS.groupby("variant")[metric_cols].mean().reset_index()

main_metrics_df["variant"] = pd.Categorical(main_metrics_df["variant"])

display_df = pd.DataFrame(
    {
        "Model": main_metrics_df["variant"],
        "Mean Train AUPR": main_metrics_df["train_cv_aupr"].round(4),
        "Mean Train F1": main_metrics_df["train_cv_f1"].round(4),
        "Mean Onset AUPR": main_metrics_df["onset_aupr"].round(4),
        "Mean Onset F1": main_metrics_df["onset_f1_class1"].round(4),
        "Mean Onset Recall": (main_metrics_df["onset_recall_class1"] * 100)
        .round(1)
        .astype(str)
        + "%",
        "Mean Onset Precision": (main_metrics_df["onset_precision_class1"] * 100)
        .round(1)
        .astype(str)
        + "%",
        "Mean Active AUPR": main_metrics_df["active_aupr"].round(4),
        "Mean Active F1": main_metrics_df["active_f1_class1"].round(4),
        "Mean Active Recall": (main_metrics_df["active_recall_class1"] * 100)
        .round(1)
        .astype(str)
        + "%",
        "Mean Active Precision": (main_metrics_df["active_precision_class1"] * 100)
        .round(1)
        .astype(str)
        + "%",
    }
)

display(display_df)

,Model,Mean Train AUPR,Mean Train F1,Mean Onset AUPR,Mean Onset F1,Mean Onset Recall,Mean Onset Precision,Mean Active AUPR,Mean Active F1,Mean Active Recall,Mean Active Precision
0,all_nopca,0.1896,0.2847,0.3341,0.3526,70.6%,23.5%,0.1630,0.2490,87.6%,14.5%
1,all_pca,0.1985,0.2856,0.3172,0.3686,60.0%,27.3%,0.2157,0.2676,62.4%,17.6%
2,conflict_nopca,0.2276,0.3236,0.3637,0.3445,33.5%,43.2%,0.1795,0.1534,19.7%,19.4%
3,conflict_pca,0.2131,0.3058,0.3566,0.3539,41.2%,34.6%,0.1970,0.2193,33.1%,19.0%
4,model_a,0.2286,0.3347,0.3390,0.3841,44.9%,34.2%,0.2079,0.2453,41.4%,18.0%


# 2.3 Final model's regional onset performance

Evaluating aggregate onset AUPR alone suggests that conflict-only text variants perform strongly. However, this aggregate metric masks critical geographic failures when broken down by region. Because the 2023 escalation was concentrated in specific hotspots, an early-warning system must successfully flag escalation in the regions where fighting actually broke out, most notably Khartoum in April 2023. The table below evaluates performance at the final configuration, pooled across all five tested seeds, focusing on recall across key escalation regions and Khartoum specifically.

Conflict-only text without PCA is not the best variant here. Across the five seeds it catches only 2 of the 25 pooled Khartoum escalation instances (a recall of 8%, in only two of the five seeds), and catches only a small fraction of key region escalations overall (recall 25.9%). All-event text without PCA does the opposite: it beats both the baseline and conflict-only models on regional recall, catching Khartoum escalations far more consistently than any other text variant.

At first glance, all-event text without PCA has the highest regional and Khartoum recall of all models, but this needs to be read against its overall predicted-positive rate, which at 68.2% is markedly higher than Model A's (30.4%) and the conflict-only variants' (20.2–29.3%). All-event text with PCA sits in between at 50.9%, closer to all-event without PCA than to the other three models, so the contrast isn't a clean two-way split, PCA reduces but does not eliminate the over-flagging tendency of the broader text corpus. All-event without PCA's key region precision (0.287) is close to its overall precision (0.235); either way, this remains the lowest precision of the five models.

This reflects the same trade-off that shows up in the aggregate onset F1 numbers (where all-event text has the stronger F1 and conflict-only text has the stronger AUPR). AUPR measures ranking quality across every possible threshold, meaning a model can score well there while still being too conservative to flag the events that matter most at its actual deployed threshold. The conflict-only models' onset recall sits well below the other models, and that shortfall is most pronounced in the exact regions where the war started.

For an early-warning use case, missing Khartoum almost entirely isn't a minor flaw, it is fundamentally failing to achieve its core purpose. On that basis, all-event text without PCA is the better candidate for onset detection specifically, even though it is the weaker performer on train-CV and aggregate AUPR.

However, this finding rests on a very small sample size of actual escalation. Because Sudan's 2023 outbreak is the only event of its kind in this dataset (5 true Khartoum escalation months, repeated identically across the five seeds), the claim that text helps at onset should be interpreted strictly as "text helped for this one escalation," rather than as a validated general property.

In [21]:
KEY_REGIONS = [
    "Khartoum",
    "North Darfur",
    "South Darfur",
    "West Darfur",
    "Central Darfur",
    "East Darfur",
    "West Kordofan",
    "South Kordofan",
]


def region_recall_table(regions, model_col="model", seed_filter=None):
    onset_pred_df = pd.read_excel(
        "evaluation/sudan_seeds_results.xlsx", sheet_name="onset_predictions"
    )

    if seed_filter is not None:
        final_config = onset_pred_df[onset_pred_df["seed"] == seed_filter]
        n_seeds = 1
    else:
        final_config = onset_pred_df
        n_seeds = final_config["seed"].nunique()

    rows = []
    for model, grp in final_config.groupby(model_col):
        key_region_rows = grp[grp["region"].isin(regions)]
        khartoum_rows = grp[grp["region"] == "Khartoum"]

        def recall_precision(sub):
            n_true = sub["y_true"].sum()
            n_pred_pos = sub["y_pred"].sum()
            n_caught = ((sub["y_true"] == 1) & (sub["y_pred"] == 1)).sum()
            recall = n_caught / n_true if n_true else float("nan")
            precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
            return n_caught, n_true, n_pred_pos, recall, precision

        n_caught, n_true, n_pred_pos, key_recall, key_precision = recall_precision(
            key_region_rows
        )
        kh_caught, kh_true, kh_pred_pos, kh_recall, kh_precision = recall_precision(
            khartoum_rows
        )

        overall_pred_pos_rate = grp["y_pred"].mean()

        rows.append(
            {
                "model": model,
                "overall_pred_positive_rate": round(overall_pred_pos_rate, 3),
                "key_regions_mean_caught": round(n_caught / n_seeds, 1),
                "key_regions_mean_true": round(n_true / n_seeds, 1),
                "key_regions_recall": round(key_recall, 3),
                "key_regions_precision": round(key_precision, 3),
                "khartoum_mean_caught": round(kh_caught / n_seeds, 1),
                "khartoum_mean_true": round(kh_true / n_seeds, 1),
                "khartoum_recall": round(kh_recall, 3),
                "khartoum_precision": round(kh_precision, 3),
            }
        )
    return pd.DataFrame(rows).set_index("model")


region_recall_table(KEY_REGIONS)

,overall_pred_positive_rate,key_regions_mean_caught,key_regions_mean_true,key_regions_recall,key_regions_precision,khartoum_mean_caught,khartoum_mean_true,khartoum_recall,khartoum_precision
model,,,,,,,,,
Model A,0.304,10.8,27.0,0.400,0.397,0.8,5.0,0.16,0.444
Model B (all-event text PCA),0.509,16.6,27.0,0.615,0.300,0.8,5.0,0.16,0.364
Model B (all-event text non-PCA),0.682,21.8,27.0,0.807,0.287,1.2,5.0,0.24,0.207
Model B (conflict-only text PCA),0.293,7.0,27.0,0.259,0.324,0.4,5.0,0.08,1.000
Model B (conflict-only text non-PCA),0.202,7.0,27.0,0.259,0.365,0.4,5.0,0.08,0.667


# 2.4 Feature importance

SHAP scores help identify why model performance varies across different conflict dynamics. To test whether text’s predictive contribution changes once war is underway, SHAP importance was computed separately for the training (2018–2022), onset (2023), and active conflict (2024–2025) periods,  averaged across the five random-search seeds at the final configuration.

Text embeddings' share of feature importance expands from training into active conflict for three of the four text variants:

* **All-event non-PCA (`all_nopca`):** Text importance rises from 60.9% in training, to 70.1% at onset, and peaks at 73.8% during active conflict.
* **Conflict-only non-PCA (`conflict_nopca`):** Text importance rises from 45.9% in training, to 49.2% at onset, and reaches 53.1% during active conflict.
* **All-event PCA (`all_pca`):** Text importance rises more modestly across all three periods, from 56.6% to 58.8% to 59.7%.

The exception is conflict-only text with PCA (`conflict_pca`), which peaks at onset (41.8%) and edges back down slightly by the active period (40.6%), rather than continuing to expand.

This pattern is broadly consistent with the active-period instability seen before (the collapse rate for all-event no-PCA rises from 12.5% at onset to 56.2% at active), as three of the four text variants come to rely more heavily on text during active conflict while Model A leans instead on its structural rolling-stat features (46.2% at active) and, more modestly, ACLED counts (32.5%). Among the text variants, only conflict-only text with PCA retains a meaningful ACLED contribution (11.9–13.1% throughout), showing this isn't simply a PCA effect, since all-event PCA crowds ACLED out too, but specifically follows from pairing a narrower, conflict-focused corpus with dimensionality reduction.

In [22]:
# TODO wordcount longer during conflict ?

In [23]:
def shap_results_fig():
    shap_results = pd.read_excel(
        "evaluation/sudan_seeds_results.xlsx", sheet_name="shap_values"
    )
    final_shap = shap_results[
        (shap_results["event"] == "sub_event")
        & (shap_results["run"].str.contains("_food_"))
    ].copy()

    shap_by_category_seed = (
        final_shap.groupby(["model", "dataset", "category", "seed"])["mean_abs_shap"]
        .sum()
        .reset_index()
    )

    shap_by_category_seed["total_SHAP"] = shap_by_category_seed.groupby(
        ["model", "dataset", "seed"]
    )["mean_abs_shap"].transform("sum")
    shap_by_category_seed["% Importance"] = (
        shap_by_category_seed["mean_abs_shap"]
        / shap_by_category_seed["total_SHAP"]
        * 100
    )

    # average the importance across the 5 seeds
    shap_by_category = (
        shap_by_category_seed.groupby(["model", "dataset", "category"])["% Importance"]
        .mean()
        .reset_index()
    )

    fig = px.bar(
        shap_by_category,
        x="model",
        y="% Importance",
        color="category",
        facet_row="dataset",
        category_orders={"dataset": ["train", "onset", "active"]},
        title="SHAP feature importance by category (Train vs Onset vs Active)",
        text_auto=".1f",
        color_discrete_sequence=px.colors.qualitative.Bold,
        height=900,
    )

    fig.update_layout(
        xaxis_title="",
        legend_title_text="Feature Category",
        hovermode="x unified",
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(t=50, b=50, l=50, r=50),
    )

    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))

    # Add light gridlines and standardise y-axis titles for all rows
    fig.update_yaxes(
        title_text="Relative Importance (%)",
    )
    fig.update_xaxes(showline=True, linewidth=1, linecolor="black")

    fig.show()

In [24]:
shap_results_fig()

# 3. Another country results to do

# 4. Final model results (best seed)
As models were trained using the best parameters using RandomizedSearchCV, there is ano guarantee that the absolute best hyperparamters have been found. Instead a comparison of the final config across five random seeds are compared. 

The table below shows that random seed 999 has the highest training AUPR for the baseline model (0.2345) and the highest mean training AUPR across all models. Therefore the seed 999 is selected for the very final comparison of one config, one seed. 

**Model performance at seed 999**

| Variant         | Train AUPR | Train F1 | Onset AUPR | Onset F1 | Active AUPR | Active F1 |
|-----------------|-----------:|---------:|-----------:|---------:|------------:|----------:|
| Model A         | 0.2345     | 0.3324   | 0.3435     | 0.4000   | 0.2106      | 0.2456    |
| all_nopca       | 0.1896     | 0.2806   | 0.3104     | 0.3422   | 0.1475      | 0.2539    |
| all_pca         | 0.2024     | 0.2894   | 0.3410     | 0.4051   | 0.2621      | 0.2963    |
| conflict_nopca  | 0.2329     | 0.3469   | 0.3229     | 0.3826   | 0.1714      | 0.2118    |
| conflict_pca    | 0.2226     | 0.3246   | 0.3500     | 0.3158   | 0.1949      | 0.1013    |

* At this seed, no text variant beats Model A on training AUPR, and only `conflict_nopca` beats it on training F1 (0.3469 vs 0.3324), consistent with the pattern found across all five seeds. 

* At onset, the two AUPR-vs-F1 split holds: `conflict_pca` beats baseline on AUPR (0.3500 vs 0.3435) but not F1, while `all_pca` beats baseline on F1 (0.4051 vs 0.4000) but not AUPR. `all_nopca` and `conflict_nopca` beat baseline on neither onset metric at this seed. 

* At active, `all_pca` is the standout, beating baseline on both AUPR (0.2621 vs 0.2106) and F1 (0.2963 vs 0.2456); `all_nopca` beats baseline on F1 only (0.2539 vs 0.2456); neither conflict-only variant beats baseline at active on either metric, with `conflict_pca` well behind on F1 (0.1013 vs 0.2456).
* At seed 999, the Khartoum result is essentially the reverse of the pooled finding: all-event text without PCA, the strongest performer pooled across seeds (0.24 recall), catches none of Khartoum's five escalations at this seed, while Model A, the weakest pooled performer (0.16), catches 2 of 5. Given Khartoum's small sample (5 true escalations per seed), this single-seed regional breakdown should not be treated as representative, the pooled 5-seed table remains the credible basis for the regional finding.

In [25]:
def seed_training_ranking(results):
    """Rank seeds by training AUPR, both for Model A alone and averaged across all five models."""

    pivot = results.pivot(
        index="seed", columns="variant", values="train_cv_aupr"
    ).round(4)
    pivot["mean_across_models"] = pivot.mean(axis=1).round(4)

    ranking = pd.DataFrame(
        {
            "model_a_train_aupr": pivot["model_a"],
            "mean_train_aupr_all_models": pivot["mean_across_models"],
        }
    ).sort_values("mean_train_aupr_all_models", ascending=False)
    ranking["model_a_rank"] = pivot["model_a"].rank(ascending=False).astype(int)
    ranking["mean_rank"] = (
        ranking["mean_train_aupr_all_models"].rank(ascending=False).astype(int)
    )

    return ranking


seed_training_ranking(PART2_RESULTS)

,model_a_train_aupr,mean_train_aupr_all_models,model_a_rank,mean_rank
seed,,,,
999,0.2345,0.2164,1,1
23,0.2331,0.2138,2,2
111,0.2295,0.2134,3,3
2025,0.2215,0.2075,5,4
32,0.2242,0.2064,4,5


In [26]:
final_config = PART2_CONFIG
final_config["seed"] = 999
FINAL_RESULTS = filter_results(ALL_RESULTS, final_config)

performance_summary(FINAL_RESULTS, ["model_a", "all_pca", "conflict_pca"])

n_configs   value
variant                                      
all_pca      active_aupr            1  0.2621
             active_f1              1  0.2963
             onset_aupr             1  0.3410
             onset_f1               1  0.4051
             train_cv_aupr          1  0.2024
             train_cv_f1            1  0.2894
conflict_pca active_aupr            1  0.1949
             active_f1              1  0.1013
             onset_aupr             1  0.3500
             onset_f1               1  0.3158
             train_cv_aupr          1  0.2226
             train_cv_f1            1  0.3246
model_a      active_aupr            1  0.2106
             active_f1              1  0.2456
             onset_aupr             1  0.3435
             onset_f1               1  0.4000
             train_cv_aupr          1  0.2345
             train_cv_f1            1  0.3324

In [27]:
region_recall_table(KEY_REGIONS, seed_filter=999)

,overall_pred_positive_rate,key_regions_mean_caught,key_regions_mean_true,key_regions_recall,key_regions_precision,khartoum_mean_caught,khartoum_mean_true,khartoum_recall,khartoum_precision
model,,,,,,,,,
Model A,0.213,9.0,27.0,0.333,0.474,2.0,5.0,0.4,0.4
Model B (all-event text PCA),0.505,18.0,27.0,0.667,0.305,0.0,5.0,0.0,NaN
Model B (all-event text non-PCA),0.639,20.0,27.0,0.741,0.282,0.0,5.0,0.0,0.0
Model B (conflict-only text PCA),0.125,2.0,27.0,0.074,0.286,0.0,5.0,0.0,NaN
Model B (conflict-only text non-PCA),0.306,12.0,27.0,0.444,0.308,1.0,5.0,0.2,1.0


# 5. Summary

### Part 1: does text improve performance? (wide 16-config grid, seed 23)

This part tests the general question across many food/rain/event-column/split combinations, all at a single seed, before narrowing to one final configuration in Part 2.

* **Train-CV:** None of the text models outperform the baseline. Conflict-only text without PCA is the least weak of four underperforming options, beating baseline on F1 in 25% of the 16 matched configurations and 12.5% on AUPR.

* **Onset (aggregate):** Conflict-only text beats baseline on AUPR in 68.8–75.0% of the 16 matched configurations but on F1 in only 12.5–25.0%. All-event text without PCA beats baseline on F1 in 87.5% of configurations, an advantage that survives excluding the 12.5% of its configurations that collapse to near-constant-positive prediction.

* **Active conflict:** Across the 16-config grid at seed 23, the baseline wins clearly for three of the four text variants, and decisively so for conflict-only non-PCA. All-event text with PCA is the exception, running close to baseline (25% AUPR win rate, 50% F1 win rate).

### Part 2: choosing and testing the final model (one configuration, averaged across 5 seeds: 23, 32, 111, 999, 2025)

This part fixes food/rain/event-column/n_splits to the single configuration justified in 2.1, and asks whether Part 1's patterns hold up at that specific configuration once seed noise is averaged out.

* **Train-CV:** None of the text variants beat Model A's mean training AUPR (0.2286) but `conflict_nopca` comes closest (0.2276), consistent with Part 1.

* **Onset:** Both conflict-only variants beat Model A on mean AUPR (`conflict_nopca` 0.3637, `conflict_pca` 0.3566 vs Model A's 0.339), consistent with Part 1. On F1, Model A leads (0.384) against a best-of-the-rest 0.369 for `all_pca`, also consistent with Part 1.

* **Onset (regional, pooled across all 5 seeds at the final config):** Conflict-only text catches only 2 of 25 pooled Khartoum escalation instances (8% recall) and around a quarter of key-region escalations. All-event text without PCA catches Khartoum and key-region escalations far more consistently than any other variant (24% and 81% recall respectively), at the cost of the lowest precision and the highest rate escalation flagging of the five models. This finding is not robust to seed choice. At seed 999, all-event text without PCA catches zero Khartoum escalations while Model A catches 2 of 5, the reverse of the pooled picture.

* **Active:** Here Part 2 diverges from Part 1. Averaged across all 5 seeds at the final config, `all_pca` edges ahead of Model A on both AUPR (0.2157 vs 0.2079) and F1, not just close to baseline as Part 1 suggested. This reversal is fragile with the seed-to-seed standard deviation in the AUPR advantage (0.026) is more than three times the mean advantage itself (0.008), and `all_pca` only wins in 2 of the 5 seeds.

### Illustrative single-seed check (seed 999, final config only, chosen on training AUPR)

At this one seed:
* `all_pca` beats baseline on both active metrics and onset F1
* `conflict_pca` beats baseline on onset AUPR only
* no variant beats baseline on training AUPR. 

This is broadly consistent with the 5-seed-averaged Part 2 picture, but the regional breakdown at this seed is not, at seed 999, all-event text without PCA catches zero Khartoum escalations, the opposite of its strongest-pooled-performer status, illustrating why the pooled regional table (not a single seed) should be treated as the credible finding.

## Overall

Given the project's objective is to evaluate whether adding text improves conflict escalation predictions, **all-event text without PCA** is the strongest candidate for onset detection specifically, both in the Part 1 grid and in the Part 2 pooled regional analysis, though this rests on a single historical escalation and is not robust to seed choice. **All-event text with PCA** is the most promising variant for active-conflict monitoring: it is the only variant that avoids clear performance decay in the Part 1 grid, and the only variant that beats baseline on average at the Part 2 final configuration, though that advantage is not yet robust across seeds. Conflict-only text, in either PCA or non-PCA form, underperforms at onset and active alike in both parts of the analysis and is hardest to justify keeping in the final model, notwithstanding its comparative strength on training and aggregate onset AUPR.